# 03 - Stage A/B Dev-Split Calibration and Checksum Freezing

This notebook implements the two-stage calibration procedure specified in §6.2 of the STP-RAG methodology plan:
1. **Stage A (Unsupervised Normalization)**: Computes closed-form median and IQR statistics for $v, a, \sigma$ strictly over the dev split.
2. **Stage B (Bayesian Optimization via Optuna)**: Tunes boundary, budget, horizon, and ranking weights targeting dev-split Recall@5.
3. **Stage C (Freezing & Checksum Verification)**: Freezes $\Theta^*$ into `configs/stp_params.yaml`, computes its SHA-256 hash, and saves it into `configs/stp_params.sha256`.

In [ ]:
from scripts.calibrate_dev_split import run_calibration, compute_file_sha256
from pathlib import Path

# Execute Stage A & Stage B calibration and Stage C freezing
calibrated_config, sha256_hash = run_calibration(
    dev_path="data/splits/dev.json",
    config_out="configs/stp_params.yaml",
    n_trials=40,
    seed=42,
)
print(f"Winning Theta frozen. Checksum: {sha256_hash}")

In [ ]:
# Verification: Assert config on disk matches the frozen checksum
cfg_path = Path("configs/stp_params.yaml")
sha_path = Path("configs/stp_params.sha256")

assert cfg_path.exists(), "configs/stp_params.yaml must exist"
assert sha_path.exists(), "configs/stp_params.sha256 must exist"

computed_hash = compute_file_sha256(cfg_path)
stored_hash = sha_path.read_text(encoding="utf-8").strip()

assert computed_hash == stored_hash, f"Checksum mismatch: {computed_hash} != {stored_hash}"
print(f"Integrity check passed! Frozen SHA-256: {computed_hash}")